In [1]:
!pip install fastapi
!pip install PyMuPDF
!pip install python-multipart
!pip install fastapi uvicorn
!pip install pyngrok

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.1/24.1 MB 61.8 MB/s eta 0:00:00


In [2]:
from fastapi import FastAPI, UploadFile, File
from typing import List
import fitz  # PyMuPDF
import joblib
import csv
import os
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import CountVectorizer

from fastapi.middleware.cors import CORSMiddleware  # ✅ جديد
from pyngrok import ngrok
import nest_asyncio
import uvicorn

app = FastAPI()

# ✅ تفعيل CORS middleware
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],  # ممكن تستبدلي "*" بـ ["https://client-dashboard.com"] بعدين
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

# ---------------- تحميل النموذج والفيكتور ----------------
MODEL_PATH = "/content/model.pkl"
VECTORIZER_PATH = "/content/vectorizer.pkl"
FEEDBACK_PATH = "feedback.csv"

if os.path.exists(MODEL_PATH) and os.path.exists(VECTORIZER_PATH):
    model = joblib.load(MODEL_PATH)
    vectorizer = joblib.load(VECTORIZER_PATH)
else:
    model = LogisticRegression()
    vectorizer = CountVectorizer()

# ---------------- استخراج النص من PDF ----------------
def extract_lines_from_pdf(file: bytes) -> List[str]:
    lines = []
    pdf_doc = fitz.open(stream=file, filetype="pdf")
    for page in pdf_doc:
        text = page.get_text()
        for line in text.split("\n"):
            line = line.strip()
            if line:
                lines.append(line)
    return lines

# ---------------- Endpoint: التوقعات ----------------
@app.post("/predict-pdf/")
async def predict_from_pdf(file: UploadFile = File(...)):
    content = await file.read()
    lines = extract_lines_from_pdf(content)

    X = vectorizer.transform(lines)
    predictions = model.predict(X)

    # ✅ بدل ما ترجّعي prediction، رجّعي النصوص بس
    return lines


# ---------------- Endpoint: حفظ الـ Feedback ----------------
@app.post("/feedback/")
async def store_feedback(data: List[dict]):
    file_exists = os.path.isfile(FEEDBACK_PATH)

    with open(FEEDBACK_PATH, mode="a", newline="", encoding="utf-8") as f:
        writer = csv.writer(f)
        if not file_exists:
            writer.writerow(["text", "true_label"])
        for item in data:
            writer.writerow([item["text"], item["true_label"]])
    return {"message": "Feedback saved."}

# ---------------- Endpoint: إعادة تدريب النموذج ----------------
@app.post("/train-feedback/")
async def train_on_feedback():
    if not os.path.isfile(FEEDBACK_PATH):
        return {"message": "No feedback data found."}

    df = pd.read_csv(FEEDBACK_PATH)
    X = vectorizer.fit_transform(df["text"])
    y = df["true_label"]

    model.fit(X, y)
    joblib.dump(model, MODEL_PATH)
    joblib.dump(vectorizer, VECTORIZER_PATH)

    return {"message": "Model retrained with feedback."}

# ---------------- ngrok + uvicorn ----------------
if __name__ == "__main__":
    ngrok.set_auth_token("2w9xGxAvHLQdIoZSShkjDooZpQz_3RhUfP2TezsP3vehcsdr")  # ← استبدله بالتوكن الخاص بك
    public_url = ngrok.connect(8000)
    print(f"✅ Public URL: {public_url}")

    nest_asyncio.apply()
    uvicorn.run(app, host="0.0.0.0", port=8000)


INFO:     Started server process [221]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


✅ Public URL: NgrokTunnel: "https://4c4da3a39258.ngrok-free.app" -> "http://localhost:8000"
INFO:     156.199.136.44:0 - "GET /docs HTTP/1.1" 200 OK
INFO:     156.199.136.44:0 - "GET /openapi.json HTTP/1.1" 200 OK


INFO:     Shutting down
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
INFO:     Finished server process [221]
